# Reference Engines
### Roadrunner propulsion reverse-engineering study · notebook 4 of 13

**Purpose.** The engine for this study is *selected* from turbojets that can actually
be bought, not designed (ADR-0023). This notebook surveys that catalogue using vendor
data only, with no cycle model, and answers:

> *What do purchasable small turbojets deliver per kilogram, per unit of frontal area
> and per kilogram of fuel, and how consistent are the vendors' own numbers?*

The figures of merit found here are the technology bounds the rest of the study
works within. The most useful one for a reverse-engineering study is **thrust per
frontal area**: once the nacelle diameter is measured from imagery (notebook 2), it
converts a diameter directly into a plausible thrust.

---
## 0 · Setup

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))   # bare-checkout fallback

from roadrunner_re import load_engine_catalogue, reports
from roadrunner_re import technology as tech
from roadrunner_re.notebook import nb_setup
from roadrunner_re.plots.engines import catalogue_overview

paths = nb_setup()

---
## 1 · The catalogue as published

All parameters are loaded from `config/reference_engines.yaml`; each entry carries
its source URL. A dash means the figure has not been retrieved from a citable
source — it is **not** filled in from memory or by estimate. Values are the vendor's
sea-level static, standard-day figures. *Mass* is with flight accessories where the
vendor states it, otherwise the bare engine.

In [ ]:
engines = load_engine_catalogue()
print(reports.catalogue_table(engines))

Several ratings are time-limited (the PBS take-off ratings are five-minute ratings),
and datasheet revisions of the same engine disagree in places; the notes in the YAML
record each case.

In [ ]:
print("Datasheet fields still to be retrieved:\n")
print(reports.missing_fields_table(engines))

---
## 2 · Figures of merit

### Theory

At maximum thrust $F$, with engine mass $m$, outer diameter $D$, air mass flow
$\dot m_a$ and fuel mass flow $\dot m_f$:

$$\frac{F}{W} = \frac{F}{m\,g_0}, \qquad
\frac{F}{A} = \frac{F}{\pi D^2/4}, \qquad
F_s = \frac{F}{\dot m_a}, \qquad
\mathrm{SFC} = \frac{\dot m_f}{F}$$

Where a vendor publishes SFC it is used directly. Otherwise it is derived from the
published fuel flow; a flow given by volume needs the fuel density
(`config/fuel.yaml`).

In [ ]:
fuel_density = tech.load_fuel_density()
metrics = tech.catalogue_metrics(engines, fuel_density.value, tech.load_gravity())
print(reports.metrics_table(metrics))

---
## 3 · Do the datasheets agree with themselves?

Five datasheets publish **both** an SFC and a fuel flow by volume. Together with the
thrust these fix the fuel density the vendor must have used:
$\rho_f = \mathrm{SFC}\cdot F / \dot V_f$. If the numbers are sound, that density
should be a plausible kerosene density and should be the same across vendors.

In [ ]:
densities = tech.implied_fuel_densities(metrics)
print(reports.implied_density_table(densities, fuel_density.value))
print(f"\nspecification range used as bounds: {fuel_density.low:.0f} to {fuel_density.high:.0f} kg/m3")

The implied densities sit within about 1 % of each other and of the configured
value. That is a real check on the catalogue: thrust, SFC and fuel flow on these
datasheets are mutually consistent, so the configured density is confirmed by the
data rather than assumed.

---
## 4 · Technology bounds

The lowest, median and highest value of each figure of merit across the catalogue.
These are the priors the later stages work within (ADR-0010); *n* is how many
engines publish enough to compute the figure.

In [ ]:
bounds = tech.technology_bounds(metrics)
print(reports.bounds_table(bounds))

---
## 5 · Engine mass against thrust

A power law $m = a\,F^{\,b}$ fitted through the catalogue. An exponent above one
means larger engines in this class are *less* weight-efficient, which is the opposite
of the usual scaling for large engines: here the larger units carry
starter-generators, oil systems and longer-life hardware that the smallest
fuel-lubricated engines do without.

In [ ]:
correlation = tech.fit_mass_correlation(metrics)
print(reports.mass_correlation_line(correlation))

In [ ]:
fig = catalogue_overview(engines, metrics, correlation, paths.figures / "reference_engines.png")

---
## 6 · Handoff

In [ ]:
handoff = tech.write_technology_handoff(
    tech.technology_handoff(engines, metrics, fuel_density), paths.out / "technology.yaml")
print(f"wrote {handoff.relative_to(paths.repo)}")

---
## Findings

- **Thrust per frontal area is the tightest figure of merit.** Across three vendors
  and a fourfold range of thrust it stays within about 20 % of its median. A measured nacelle
  diameter therefore bounds the thrust of whatever engine sits inside it far better
  than any other public evidence can.
- **Thrust-to-weight falls with size** in this class, roughly from twelve for the
  smallest engines to six for the largest, on a with-accessories basis.
- **SFC improves with size**, by roughly a quarter from the smallest to the largest
  engines. All of it is poor by large-engine standards, which is what makes loiter
  expensive for this vehicle.
- **Idle thrust is three to five per cent of maximum** where published, so the
  vertical-landing thrust floor (ADR-0009) is unlikely to bind.
- **Compressor pressure ratio is published for only four engines**, all close to
  four. That bound is weak.
- **Standing finding:** masses are not on a common basis across vendors, and several
  datasheet fields are still missing (listed in section 1).